# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIRˆ² clinical oncology dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library with a Croissant schema.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant Schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset via Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # DatasetMetadata object

print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets by their @id and name
print("Available RecordSets:")
for rs in metadata.record_sets:
    print(f"  - @id: {rs.id} | name: {rs.name if hasattr(rs,'name') else ''}")

# For the first RecordSet, list its fields and their @id
if metadata.record_sets:
    record_set = metadata.record_sets[0]
    print(f"\nRecordSet selected: {record_set.id}")
    print("Fields in this record set:")
    for f in record_set.fields:
        print(f"  - @id: {f.id} | name: {f.name if hasattr(f,'name') else ''} | dataType: {f.data_type if hasattr(f,'data_type') else ''}")

## 3. Data Extraction

Load data from all available record sets into pandas DataFrames for analysis. Use record set and field `@id`s noted in the overview.

In [ ]:
# List of record set @id's (using ids from previous overview)
record_sets_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_sets_ids:
    # Load records from each record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records from RecordSet {record_set_id}.")
    else:
        print(f"No records loaded for RecordSet {record_set_id}.")

# For demonstration, use the first non-empty DataFrame
selected_rs_id = next((k for k, v in dataframes.items() if not v.empty), None)
if selected_rs_id is not None:
    print(f"\nColumns in selected RecordSet ({selected_rs_id}):\n", dataframes[selected_rs_id].columns.tolist())
    display(dataframes[selected_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

We will select a numeric field for data manipulation, such as filtering, normalization, and grouping.


In [ ]:
# Choose a numeric field for EDA

# Get the first record set for which we have a non-empty dataframe
record_set_id = selected_rs_id
df = dataframes[record_set_id] if record_set_id is not None else None

# List fields and infer numeric ones
numeric_field_id = None
group_field_id = None
if record_set_id and df is not None:
    record_set = next(rs for rs in metadata.record_sets if rs.id == record_set_id)
    print("\nAll fields by @id, name, type:")
    for f in record_set.fields:
        print(f"  - @id: {f.id} | name: {getattr(f,'name',None)} | dataType: {getattr(f,'data_type',None)}")
    # Look for integer/float fields
    numeric_fields = [f for f in record_set.fields if getattr(f, 'data_type', None) in ('Integer', 'Float', 'Number', 'schema:Integer', 'schema:Float', 'schema:Number')]
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
        print(f"\nUsing numeric field for filtering/normalization: {numeric_field_id}")
    # Look for a categorical/text group field
    text_fields = [f for f in record_set.fields if getattr(f, 'data_type', None) in ('Text', 'schema:Text', 'String')]
    if text_fields:
        group_field_id = text_fields[0].id
        print(f"Grouping on: {group_field_id}")

    # EDA: filter, normalize, group
    if numeric_field_id is not None and numeric_field_id in df.columns:
        # Ensure column is numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std if std else 0
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if (df is not None) and (numeric_field_id is not None and numeric_field_id in df.columns):
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If group_field is available, boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        df.boxplot(column=numeric_field_id, by=group_field_id, rot=90)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

- We loaded the FAIRˆ² colorectal cancer survivor dataset using Croissant and explored its schema programmatically via `mlcroissant`.
- All data loading and referencing was done via entity `@id`, improving reproducibility and schema alignment.
- We performed sample EDA, including filtering, normalization, grouping, and visualization of numeric attributes.
  
Further analyses may involve richer statistical exploration, survival modeling, and clinical outcome stratification using these data.
